# OmniFinance tributário (Colab + GPU)

Serve o `rwayz/tributario-llama-8b-v2` — `Llama-3.1-8B-Instruct` com fine-tune completo em tributário brasileiro — em 4-bit como uma API HTTP acessível pelo projeto Python local.

**Antes de rodar:** `Ambiente de execução > Alterar o tipo de ambiente de execução > T4 GPU`.
**Não precisa de token:** o repositório é público (licença Llama 3.1).

Rode as células em ordem. A última imprime `COLAB_LLM_URL=...`, que vai para o `.env` do projeto.

In [ ]:
!nvidia-smi -L || echo "SEM GPU: troque o tipo de ambiente para T4 GPU"

In [ ]:
%pip -q install -U "transformers>=4.48" accelerate bitsandbytes fastapi uvicorn
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared && cloudflared --version

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELO = "rwayz/tributario-llama-8b-v2"

tokenizer = AutoTokenizer.from_pretrained(MODELO)

# 4-bit: os 16 GB em bf16 nao cabem com folga nos 16 GB da T4
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 (Turing) nao tem bfloat16
)
modelo = AutoModelForCausalLM.from_pretrained(
    MODELO, quantization_config=bnb, device_map="auto"
).eval()
DISPOSITIVO = str(next(modelo.parameters()).device)

print(f"{MODELO} carregado em {DISPOSITIVO} (4-bit)")

In [ ]:
import json

def extrair_chamada(texto: str) -> dict | None:
    """Le o JSON de function call que o Llama-3.1 emite e normaliza."""
    if "<|python_tag|>" in texto:
        texto = texto.split("<|python_tag|>", 1)[1]
    inicio = texto.find("{")
    if inicio < 0:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(texto[inicio:])
    except ValueError:
        return None
    if not isinstance(obj, dict) or "name" not in obj:
        return None
    argumentos = obj.get("parameters") or obj.get("arguments") or {}
    if not isinstance(argumentos, dict):
        return None
    return {"name": obj["name"], "arguments": argumentos}


@torch.inference_mode()
def responder(
    mensagens: list[dict],
    ferramentas: list[dict] | None = None,
    max_new_tokens: int = 512,
) -> dict:
    """Recebe o historico no formato do chat template e devolve texto ou tool call."""
    entrada = tokenizer.apply_chat_template(
        mensagens,
        tools=ferramentas,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(DISPOSITIVO)
    saida = modelo.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    texto = tokenizer.decode(
        saida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    return {
        "texto": texto,
        "tool_call": extrair_chamada(texto),
        "modelo": MODELO,
        "dispositivo": DISPOSITIVO,
    }


responder(
    [{"role": "user", "content": "O que e juro composto?"}], max_new_tokens=64
)

In [ ]:
import threading
import traceback

import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="OmniFinance Tributario")


class ChatEntrada(BaseModel):
    mensagens: list[dict]
    ferramentas: list[dict] | None = None
    max_new_tokens: int = 512


@app.get("/health")
def health() -> dict:
    return {"ok": True, "modelo": MODELO, "dispositivo": DISPOSITIVO}


@app.post("/chat")
def rota_chat(entrada: ChatEntrada) -> dict:
    try:
        return responder(entrada.mensagens, entrada.ferramentas, entrada.max_new_tokens)
    except Exception as exc:
        traceback.print_exc()
        return {
            "erro": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc()[-1500:],
        }


threading.Thread(
    target=lambda: uvicorn.run(
        app, host="0.0.0.0", port=8000, log_level="warning"
    ),
    daemon=True,
).start()
print("API ouvindo em http://localhost:8000")


In [ ]:
import re
import subprocess
import time

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,
)

urls: list[str] = []


def _ler_saida() -> None:
    for linha in proc.stderr:
        achado = re.search(r"https://[-\w]+\.trycloudflare\.com", linha)
        if achado:
            urls.append(achado.group(0))


threading.Thread(target=_ler_saida, daemon=True).start()

for _ in range(60):
    if urls:
        break
    time.sleep(1)

print("COLAB_LLM_URL=" + (urls[0] if urls else "FALHOU - veja o log do cloudflared"))

Copie a linha `COLAB_LLM_URL=...` para o `.env` do projeto:

```env
COLAB_LLM_URL=https://algo-assim.trycloudflare.com
```

Teste rápido do lado local:

```bash
python scripts/testar_colab.py
```

**A URL morre quando esta sessão do Colab cai** (90 min ocioso na conta gratuita, 12 h no total). O cloudflared roda enquanto o notebook estiver executando — nunca pare a última célula. Se a sessão cair, rode tudo de novo e atualize o `.env`.